# Module 6: Functions in Python

## From Basic Definitions to Functional Programming

---

### Learning Objectives

By the end of this module, you will be able to:
- Define and call functions with various parameter types
- Use default arguments, `*args`, and `**kwargs`
- Write lambda functions and use `map`/`filter`/`reduce`
- Understand variable scope (LEGB rule)
- Implement recursive algorithms
- Write proper docstrings and type hints
- Build reusable bioinformatics utility functions

---

**Why functions matter in bioinformatics:** Every analysis pipeline is built from functions -- GC content calculators, sequence translators, file parsers, statistical tests. Writing clean, reusable functions is what separates a quick script from a maintainable tool.

## Why this notebook matters

Functions are the most important abstraction in programming. Without them, every bioinformatics script is a monolithic wall of code that cannot be tested, reused, or shared. With well-designed functions, your `gc_content()` can be called from a FASTA parser, a batch quality-check script, and a Jupyter notebook without any changes. This notebook covers everything from the basics (`def`, `return`) to advanced patterns (`*args`, `**kwargs`, closures, decorators-lite) using biological examples throughout.

## How to use this notebook

1. Run cells in order. Sections 1–4 cover essentials (definition, parameters, defaults, docstrings). Sections 5–9 cover advanced topics. Sections 10–11 provide complete examples and exercises.
2. Each exercise has a blank cell (write your solution) followed by a solution cell. **Do the blank one first** — the exercises test real bioinformatics tasks.
3. The scope section (Section 7) is conceptually important: understand it before writing functions that modify global state.

## Common stumbling points

- **`return` vs `print()`:** A function that uses `print()` sends output to the screen but returns `None`. If you want to use the result elsewhere, use `return`. Beginners often call `result = my_function()` and get `None` because they forgot `return`.
- **Mutable default arguments are a trap:** `def f(x, data=[])` is a classic Python bug. The default list is created *once* when the function is defined, not each time it is called. On the second call, the list from the first call is still there. Always use `None` as default for mutable arguments: `def f(x, data=None): if data is None: data = []`.
- **Variable scope (LEGB):** A variable inside a function is local — it does not exist outside the function. A function can *read* a global variable but cannot reassign it without `global`. This is intentional: it prevents functions from silently breaking each other.
- **`*args` and `**kwargs` order matters:** The order must be: required args, default args, `*args`, keyword-only args, `**kwargs`. Getting this wrong causes a `SyntaxError`.
- **Lambda functions cannot contain statements:** A `lambda` can only contain a single *expression* — no `if` blocks, no `for` loops, no assignment. Use `def` for anything more complex.

---

[← Previous: Module 5: Control Flow](../05_Control_Flow/01_control_flow.ipynb) | [Next: Module 7: File Operations →](../07_File_Operations/01_file_operations.ipynb)

---

## 1. Function Basics

A function is a reusable block of code that you can call from anywhere in your program.

```python
def function_name(parameter1, parameter2):
    """Docstring: describes what the function does."""
    # function body
    return result
```

**Why use functions?**
- Write code once, call it many times (DRY: Don't Repeat Yourself)
- Give a name to a block of logic — easier to read
- Test in isolation
- Share with others (as a module)

In [ ]:
# Simplest function: no parameters, no return value
def greet():
    print("Hello, Bioinformatician!")

greet()
greet()  # call it as many times as you like

In [ ]:
# Function with a parameter and a return value
def calculate_gc_content(sequence):
    """Calculate GC content of a DNA sequence.
    
    Args:
        sequence: DNA sequence string (ATGC characters)
    
    Returns:
        GC content as a percentage (float)
    """
    sequence = sequence.upper()
    gc_count = sequence.count('G') + sequence.count('C')
    return (gc_count / len(sequence)) * 100

# Call the function
seq = "ATGCGATCGATCGTAGC"
gc = calculate_gc_content(seq)
print(f"GC content of {seq}: {gc:.2f}%")

In [ ]:
# Functions can return multiple values as a tuple
def analyze_nucleotides(sequence):
    """Count all nucleotides in a DNA sequence."""
    seq = sequence.upper()
    return seq.count('A'), seq.count('T'), seq.count('G'), seq.count('C')

# Unpack the returned tuple
a, t, g, c = analyze_nucleotides("ATGCGATCGATCGATCG")
print(f"A={a}, T={t}, G={g}, C={c}")
print(f"Total: {a + t + g + c} nucleotides")

In [ ]:
# Returning a dictionary for richer results
def analyze_sequence(sequence):
    """Comprehensive sequence analysis.
    
    Returns:
        dict with length, gc_content, and nucleotide counts
    """
    seq = sequence.upper()
    length = len(seq)
    gc = (seq.count('G') + seq.count('C')) / length * 100 if length > 0 else 0
    counts = {nuc: seq.count(nuc) for nuc in 'ATGC'}
    
    return {'length': length, 'gc_content': round(gc, 2), 'counts': counts}

result = analyze_sequence("ATGCGATCGATCGTAGCGATCGATCG")
print(f"Length: {result['length']} bp")
print(f"GC content: {result['gc_content']}%")
print(f"Nucleotide counts: {result['counts']}")

---
## 2. Default Arguments

Parameters can have default values. Required arguments must come before defaults.

```
Positional:  def func(a, b)       -> func(1, 2)
Keyword:     def func(a, b)       -> func(a=1, b=2)
Default:     def func(a, b=10)    -> func(1)  # b defaults to 10
```

In [ ]:
import random

def make_random_sequence(length=200, alphabet="ATGC"):
    """Generate a random nucleotide sequence.
    
    Args:
        length: Number of nucleotides (default: 200)
        alphabet: Characters to choose from (default: 'ATGC' for DNA)
    """
    return ''.join(random.choice(alphabet) for _ in range(length))

# Use defaults
dna = make_random_sequence()
print(f"Random DNA (200 bp): {dna[:50]}...")

# Override length
short_dna = make_random_sequence(length=20)
print(f"Short DNA (20 bp): {short_dna}")

# Generate RNA instead
rna = make_random_sequence(length=20, alphabet="AUGC")
print(f"Random RNA (20 bp): {rna}")

In [ ]:
# Codon translation with default table
def translate_codon(codon, table='standard'):
    """Translate a DNA codon to its amino acid."""
    standard_table = {
        'ATG': 'M', 'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
        'TAA': '*', 'TAG': '*', 'TGA': '*',
        'GGG': 'G', 'GGA': 'G', 'GGC': 'G', 'GGT': 'G',
        'AAA': 'K', 'AAG': 'K', 'GCG': 'A', 'TGG': 'W',
    }
    return standard_table.get(codon.upper(), 'X')  # X for unknown

print(f"ATG -> {translate_codon('ATG')}  (Methionine / Start)")
print(f"TAA -> {translate_codon('TAA')}  (Stop)")
print(f"GGC -> {translate_codon('GGC')}  (Glycine)")

In [ ]:
# WARNING: Mutable default arguments are a classic Python trap!

# BAD -- the default list is created ONCE and shared across calls
def bad_collect(item, items=[]):
    items.append(item)
    return items

print(bad_collect('A'))  # ['A']
print(bad_collect('T'))  # ['A', 'T'] -- Surprise! The list persists!

print()

# GOOD -- use None and create a fresh list each call
def good_collect(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items

print(good_collect('A'))  # ['A']
print(good_collect('T'))  # ['T'] -- Correct!

---
## 3. `*args` and `**kwargs`

Handle variable numbers of arguments:
- `*args` collects extra **positional** arguments into a **tuple**
- `**kwargs` collects extra **keyword** arguments into a **dictionary**

In [ ]:
# *args: concatenate any number of sequences
def concatenate_sequences(*sequences):
    """Join any number of DNA/RNA sequences together."""
    return ''.join(sequences)

print(concatenate_sequences('ATG', 'CCC', 'GGG'))  # 'ATGCCCGGG'
print(concatenate_sequences('ATG'))                 # 'ATG'

In [ ]:
# **kwargs: build a FASTA header with flexible metadata
def create_fasta_header(sequence_id, **metadata):
    """Create a FASTA header line from an ID and arbitrary metadata."""
    header = f">{sequence_id}"
    for key, value in metadata.items():
        header += f" [{key}={value}]"
    return header

header = create_fasta_header(
    "BRCA1_human",
    organism="Homo sapiens",
    gene="BRCA1",
    length=5500,
    chromosome=17
)
print(header)

In [ ]:
# Combining all argument types
# Order: required, default, *args, keyword-only, **kwargs

def sequence_stats(*sequences, normalize=False, **options):
    """Calculate statistics for multiple sequences.
    
    Args:
        *sequences: Variable number of DNA sequences
        normalize: If True, report nucleotide fractions instead of counts
        **options: Additional options (e.g., include_gc=True)
    """
    include_gc = options.get('include_gc', True)
    results = []
    
    for seq in sequences:
        seq = seq.upper()
        stats = {'length': len(seq)}
        for nuc in 'ATGC':
            count = seq.count(nuc)
            stats[nuc] = count / len(seq) if normalize else count
        if include_gc:
            stats['gc_content'] = (seq.count('G') + seq.count('C')) / len(seq) * 100
        results.append(stats)
    
    return results

stats = sequence_stats('ATGCATGC', 'GGGGCCCC', normalize=True, include_gc=True)
for i, s in enumerate(stats):
    print(f"Sequence {i+1}: {s}")

---
## 4. Docstrings and Type Hints

Good documentation makes your functions usable by others (and by future you).

In [ ]:
def validate_dna_sequence(sequence: str, allow_n: bool = False) -> bool:
    """Check whether a string is a valid DNA sequence.
    
    Args:
        sequence: The string to validate.
        allow_n: If True, allow 'N' as an ambiguous nucleotide.
    
    Returns:
        True if the sequence contains only valid DNA characters.
    
    Examples:
        >>> validate_dna_sequence('ATGC')
        True
        >>> validate_dna_sequence('ATXG')
        False
        >>> validate_dna_sequence('ATNG', allow_n=True)
        True
    """
    valid_chars = set('ATGCN') if allow_n else set('ATGC')
    return set(sequence.upper()) <= valid_chars

# Test
print(validate_dna_sequence('ATGCGATCG'))          # True
print(validate_dna_sequence('ATGXYZ'))              # False
print(validate_dna_sequence('ATGNNNC', allow_n=True))  # True

In [ ]:
def find_orfs(sequence: str, min_length: int = 30) -> list[dict]:
    """Find all Open Reading Frames (ORFs) in a DNA sequence.
    
    An ORF starts with ATG and ends with a stop codon (TAA, TAG, TGA).
    
    Args:
        sequence: DNA sequence to search.
        min_length: Minimum ORF length in nucleotides (default: 30).
    
    Returns:
        List of dicts with 'start', 'end', 'length', and 'sequence' keys.
    """
    sequence = sequence.upper()
    stop_codons = {'TAA', 'TAG', 'TGA'}
    orfs = []
    
    for frame in range(3):  # Three reading frames
        i = frame
        while i < len(sequence) - 2:
            codon = sequence[i:i+3]
            if codon == 'ATG':
                # Found a start codon -- scan for stop
                for j in range(i + 3, len(sequence) - 2, 3):
                    next_codon = sequence[j:j+3]
                    if next_codon in stop_codons:
                        orf_seq = sequence[i:j+3]
                        if len(orf_seq) >= min_length:
                            orfs.append({
                                'start': i,
                                'end': j + 3,
                                'length': len(orf_seq),
                                'sequence': orf_seq
                            })
                        i = j + 3  # Continue after this stop codon
                        break
                else:
                    i += 3
                    continue
                continue
            i += 3
    
    return sorted(orfs, key=lambda x: x['start'])

# Test: a sequence containing two ORFs
test_seq = "CCCATGGCCGATCGATAGCCCATGAAAGGGCCCTTTAAATTT"
orfs = find_orfs(test_seq, min_length=9)
for orf in orfs:
    print(f"ORF at {orf['start']}-{orf['end']}: {orf['sequence']} ({orf['length']} bp)")

---
## 5. Lambda Functions

Anonymous, single-expression functions. Useful as short callbacks for sorting/filtering.

```python
lambda arguments: expression
```

**Important:** Lambdas provide no speed advantage over regular functions. They are purely a convenience for short, throwaway operations.

In [ ]:
# Regular function vs. lambda
def gc_content(seq):
    s = seq.upper()
    return (s.count('G') + s.count('C')) / len(s) * 100

gc_lambda = lambda seq: (seq.upper().count('G') + seq.upper().count('C')) / len(seq) * 100

test = "ATGCGATCGATCG"
print(f"Function: {gc_content(test):.1f}%")
print(f"Lambda:   {gc_lambda(test):.1f}%")

In [ ]:
# Lambdas shine when used with sorted(), max(), min()
genes = [
    {'name': 'BRCA1', 'length': 7088, 'gc': 42.3},
    {'name': 'TP53',  'length': 2512, 'gc': 51.2},
    {'name': 'EGFR',  'length': 5616, 'gc': 48.7},
    {'name': 'MYC',   'length': 2357, 'gc': 55.1},
]

# Sort by GC content (descending)
for g in sorted(genes, key=lambda g: g['gc'], reverse=True):
    print(f"  {g['name']:6s}: {g['gc']}% GC, {g['length']} bp")

In [ ]:
# Lambda for sorting dictionary keys by their values
# (common pattern from the original Russian course material)
codon_usage = {'ATG': 100, 'GGC': 53, 'AAA': 31, 'TTT': 42}

sorted_codons = sorted(codon_usage, key=lambda codon: codon_usage[codon])
print("Codons sorted by frequency (ascending):")
for codon in sorted_codons:
    print(f"  {codon}: {codon_usage[codon]}")

---
## 6. `map`, `filter`, and `reduce`

Functional programming tools that apply functions to sequences of data.

```
map(func, iterable)     ->  Apply func to each element
filter(func, iterable)  ->  Keep elements where func returns True
reduce(func, iterable)  ->  Combine elements pairwise into one value
```

In [ ]:
# MAP: apply a function to every element
sequences = ['atgc', 'GATC', 'AtGc', 'ttaa']

upper_seqs = list(map(str.upper, sequences))
print(f"Uppercase: {upper_seqs}")

lengths = list(map(len, sequences))
print(f"Lengths:   {lengths}")

# Practical: convert a line of numbers (common in bioinformatics input)
numbers_str = "1 3 4 77 82"
numbers = list(map(int, numbers_str.split()))
print(f"Parsed numbers: {numbers}")

In [ ]:
# FILTER: keep only GC-rich sequences (>50% GC)
sequences = ["ATATATAT", "GCGCGCGC", "ATGCATGC", "AAAGGGCC", "GCGATCGC"]

gc_rich = list(filter(
    lambda s: (s.count('G') + s.count('C')) / len(s) > 0.5,
    sequences
))
print(f"GC-rich sequences (>50%): {gc_rich}")

In [ ]:
from functools import reduce

# REDUCE: concatenate all exon sequences into mRNA
exons = ["ATG", "GCCGAT", "CGATAG"]
mrna = reduce(lambda a, b: a + b, exons)
print(f"mRNA from exons: {mrna}")

# REDUCE: find the longest sequence in a dataset
sequences = ["ATGC", "GATCGATC", "AT", "GCGCGCGCGC"]
longest = reduce(lambda a, b: a if len(a) > len(b) else b, sequences)
print(f"Longest sequence: {longest} ({len(longest)} bp)")

---
## 7. Variable Scope (LEGB Rule)

When Python encounters a variable name, it searches in this order:

1. **L**ocal -- inside the current function
2. **E**nclosing -- in any outer/enclosing functions
3. **G**lobal -- at the module level
4. **B**uilt-in -- Python's built-in names (`len`, `print`, `range`, ...)

In [ ]:
# Local vs. global
organism = "E. coli"  # Global scope

def show_organism():
    organism = "H. sapiens"  # Local scope -- shadows the global
    print(f"Inside function: {organism}")

show_organism()
print(f"Outside function: {organism}")  # Global is unchanged

In [ ]:
# The 'global' keyword lets a function modify a global variable
# (use sparingly -- it makes code harder to reason about)
sample_count = 0

def process_sample():
    global sample_count
    sample_count += 1

process_sample()
process_sample()
print(f"Processed {sample_count} samples")

In [ ]:
# Enclosing scope and closures
# A closure is a function that remembers variables from its enclosing scope

def make_gc_filter(threshold):
    """Return a function that filters sequences by GC content."""
    def passes_filter(sequence):
        gc = (sequence.count('G') + sequence.count('C')) / len(sequence) * 100
        return gc >= threshold  # 'threshold' comes from enclosing scope
    return passes_filter

# Create specialized filters
is_gc_rich = make_gc_filter(60)
is_moderate_gc = make_gc_filter(40)

test_seqs = ["GCGCGCGC", "ATATATAT", "ATGCATGC"]
for seq in test_seqs:
    print(f"{seq}: GC-rich={is_gc_rich(seq)}, moderate-GC={is_moderate_gc(seq)}")

In [ ]:
# The 'nonlocal' keyword: modify a variable in enclosing scope
# Useful for counters and accumulators in nested functions

def get_sequence_counter(start_id=0):
    """Return a counter function that generates unique sequence IDs."""
    count = start_id - 1
    def next_id():
        nonlocal count
        count += 1
        return f"seq_{count:04d}"
    return next_id

counter = get_sequence_counter(1)
print(counter())  # seq_0001
print(counter())  # seq_0002
print(counter())  # seq_0003

---
## 8. Recursion

A function that calls itself. Every recursive function needs:
1. **Base case** -- when to stop
2. **Recursive case** -- how to reduce the problem

```
factorial(5)
  -> 5 * factorial(4)
        -> 4 * factorial(3)
              -> 3 * factorial(2)
                    -> 2 * factorial(1)
                          -> return 1  (base case)
                    -> return 2
              -> return 6
        -> return 24
  -> return 120
```

In [ ]:
# Factorial: iterative vs. recursive
def factorial_iterative(n):
    acc = 1
    for i in range(1, n + 1):
        acc *= i
    return acc

def factorial_recursive(n):
    if n <= 1:  # Base case
        return 1
    return n * factorial_recursive(n - 1)  # Recursive case

print(f"5! (iterative) = {factorial_iterative(5)}")
print(f"5! (recursive) = {factorial_recursive(5)}")

In [ ]:
# Fibonacci: rabbit population modeling!
# In 1202, Leonardo of Pisa (Fibonacci) modeled rabbit populations:
# Each pair produces a new pair every month, starting after one month of maturity.
# Month:      1  2  3  4  5  6  7   8   9  10
# Pairs:      1  1  2  3  5  8  13  21  34  55

def fibonacci_recursive(n):
    """Calculate nth Fibonacci number.
    WARNING: Very slow for large n due to exponential repeated calculations!
    """
    if n <= 0:
        return 0
    if n == 1:
        return 1
    return fibonacci_recursive(n - 1) + fibonacci_recursive(n - 2)

def fibonacci_iterative(n):
    """Fast iterative Fibonacci."""
    f0, f1 = 0, 1
    for _ in range(n):
        f0, f1 = f1, f0 + f1
    return f0

print("Fibonacci (rabbit population over 10 months):")
print([fibonacci_iterative(i) for i in range(10)])

In [ ]:
# Memoization: caching results to avoid recomputation
# This exploits the mutable default argument "trick" intentionally

def fibonacci_memoized(n, cache={0: 0, 1: 1}):
    """Recursive Fibonacci with memoization via mutable default."""
    if n in cache:
        return cache[n]
    result = fibonacci_memoized(n - 1) + fibonacci_memoized(n - 2)
    cache[n] = result
    return result

# Now it's fast even for large n
print(f"fibonacci_memoized(50) = {fibonacci_memoized(50)}")

In [ ]:
# Recursion has limits: Python's default call stack is ~1000 frames
import sys
print(f"Default recursion limit: {sys.getrecursionlimit()}")

# For deep recursion, use iterative approaches or increase the limit
# sys.setrecursionlimit(5000)  # Use with caution

---
## 9. `functools` Utilities

The `functools` module provides higher-order functions that act on or return other functions.

In [ ]:
from functools import lru_cache
import time

# lru_cache: automatic memoization (better than manual memoization)
@lru_cache(maxsize=128)
def fibonacci_cached(n):
    """Fibonacci with automatic caching."""
    if n < 2:
        return n
    return fibonacci_cached(n - 1) + fibonacci_cached(n - 2)

start = time.perf_counter()
result = fibonacci_cached(100)
elapsed = time.perf_counter() - start
print(f"fibonacci_cached(100) = {result}")
print(f"Time: {elapsed:.6f} seconds")
print(f"Cache info: {fibonacci_cached.cache_info()}")

In [ ]:
from functools import partial

# partial: create specialized versions of functions
def calculate_similarity(seq1, seq2, match_score=1, mismatch_penalty=-1):
    """Simple position-by-position similarity score."""
    score = 0
    for a, b in zip(seq1, seq2):
        score += match_score if a == b else mismatch_penalty
    return score

# Create specialized scoring functions
strict_scoring = partial(calculate_similarity, match_score=2, mismatch_penalty=-2)
lenient_scoring = partial(calculate_similarity, match_score=1, mismatch_penalty=0)

s1, s2 = "ATGCATGC", "ATGAATGC"
print(f"Standard: {calculate_similarity(s1, s2)}")
print(f"Strict:   {strict_scoring(s1, s2)}")
print(f"Lenient:  {lenient_scoring(s1, s2)}")

---
## 10. Putting It All Together: Bioinformatics Toolkit

Let us combine what we have learned into a complete sequence analysis toolkit.

In [ ]:
# Complete codon table
CODON_TABLE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

def transcribe(dna: str) -> str:
    """Transcribe DNA to RNA (T -> U)."""
    return dna.upper().replace('T', 'U')

def translate(dna: str) -> str:
    """Translate a DNA sequence to a protein string.
    
    Translates from the first nucleotide. Stops at the first stop codon.
    """
    dna = dna.upper()
    protein = []
    for i in range(0, len(dna) - 2, 3):
        codon = dna[i:i+3]
        aa = CODON_TABLE.get(codon, 'X')
        if aa == '*':
            break
        protein.append(aa)
    return ''.join(protein)

def reverse_complement(sequence: str) -> str:
    """Return the reverse complement of a DNA sequence."""
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    return ''.join(complement.get(n, 'N') for n in sequence.upper()[::-1])

# Central Dogma demonstration
dna = "ATGGCCGATCGATAG"
print(f"DNA:                5'-{dna}-3'")
print(f"Reverse complement: 3'-{reverse_complement(dna)}-5'")
print(f"mRNA:               5'-{transcribe(dna)}-3'")
print(f"Protein:            {translate(dna)}")
print(f"\nGC content: {calculate_gc_content(dna):.1f}%")

In [ ]:
# Melting temperature calculator for PCR primers
def melting_temperature(primer: str) -> float:
    """Estimate melting temperature (Tm) of a DNA primer.
    
    Uses the basic formula for primers < 14 bp:
        Tm = 4*(G+C) + 2*(A+T)
    And the Wallace rule for longer primers:
        Tm = 64.9 + 41*(G+C - 16.4) / (A+T+G+C)
    
    Args:
        primer: DNA sequence of the primer
    
    Returns:
        Estimated Tm in degrees Celsius
    """
    primer = primer.upper()
    g = primer.count('G')
    c = primer.count('C')
    a = primer.count('A')
    t = primer.count('T')
    
    if len(primer) < 14:
        return 4 * (g + c) + 2 * (a + t)
    else:
        return 64.9 + 41 * (g + c - 16.4) / (a + t + g + c)

primers = ["ATGCGATCG", "GCGCGCGCGCGCGCGCGCGC", "AATTAATTAATTAA"]
for p in primers:
    tm = melting_temperature(p)
    print(f"Primer: {p:25s}  Tm = {tm:.1f} C")

---
## 11. Practice Exercises

### Exercise 1: Reverse Complement Function

Write a function that takes a DNA sequence and returns its reverse complement. Handle both uppercase and lowercase input. Include a docstring with type hints.

In [ ]:
# Exercise 1: Write your function here

def reverse_complement_ex(sequence: str) -> str:
    """Return the reverse complement of a DNA sequence.
    
    Args:
        sequence: DNA sequence (case-insensitive)
    
    Returns:
        Reverse complement in uppercase
    """
    # YOUR CODE HERE
    pass

# Test cases:
# reverse_complement_ex("ATGC") should return "GCAT"
# reverse_complement_ex("aattcc") should return "GGAATT"
# reverse_complement_ex("AAAA") should return "TTTT"

In [ ]:
# Exercise 1: SOLUTION

def reverse_complement_ex(sequence: str) -> str:
    """Return the reverse complement of a DNA sequence.
    
    Args:
        sequence: DNA sequence (case-insensitive)
    
    Returns:
        Reverse complement in uppercase
    """
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    return ''.join(complement.get(n, 'N') for n in sequence.upper()[::-1])

# Verify
assert reverse_complement_ex("ATGC") == "GCAT"
assert reverse_complement_ex("aattcc") == "GGAATT"
assert reverse_complement_ex("AAAA") == "TTTT"
print("All tests passed!")

### Exercise 2: Hamming Distance

The Hamming distance between two strings of equal length is the number of positions at which the corresponding characters differ. This is used to measure mutation distance between DNA sequences.

In [ ]:
# Exercise 2: Write your function here

def hamming_distance(seq1: str, seq2: str) -> int:
    """Calculate the Hamming distance between two sequences.
    
    Args:
        seq1: First DNA sequence
        seq2: Second DNA sequence (same length as seq1)
    
    Returns:
        Number of positions where the sequences differ
    
    Raises:
        ValueError: If the sequences have different lengths
    """
    # YOUR CODE HERE
    pass

# Test cases:
# hamming_distance("ATGC", "ATGC") should return 0
# hamming_distance("ATGC", "ATGG") should return 1
# hamming_distance("AAAA", "TTTT") should return 4

In [ ]:
# Exercise 2: SOLUTION

def hamming_distance(seq1: str, seq2: str) -> int:
    """Calculate the Hamming distance between two sequences.
    
    Args:
        seq1: First DNA sequence
        seq2: Second DNA sequence (same length as seq1)
    
    Returns:
        Number of positions where the sequences differ
    
    Raises:
        ValueError: If the sequences have different lengths
    """
    if len(seq1) != len(seq2):
        raise ValueError(f"Sequences must be the same length ({len(seq1)} != {len(seq2)})")
    return sum(a != b for a, b in zip(seq1.upper(), seq2.upper()))

# Verify
assert hamming_distance("ATGC", "ATGC") == 0
assert hamming_distance("ATGC", "ATGG") == 1
assert hamming_distance("AAAA", "TTTT") == 4
assert hamming_distance("GAGCCTACTAACGGGAT", "CATCGTAATGACGGCCT") == 7
print("All tests passed!")

# Biological example: compare homologous sequences
human_seq  = "ATGGCCGATCGATCGATCGATCG"
mouse_seq  = "ATGGCCGATCAATCGATCAATCG"
d = hamming_distance(human_seq, mouse_seq)
pct = (1 - d / len(human_seq)) * 100
print(f"\nHuman vs Mouse: {d} mutations, {pct:.1f}% identity")

### Exercise 3: Sequence Similarity Score

Write a function that computes a position-based similarity score between two sequences, with customizable scoring. The function should handle sequences of different lengths (align from the start, ignore overhanging bases).

In [ ]:
# Exercise 3: Write your function here

def similarity_score(seq1: str, seq2: str, 
                     match: int = 1, 
                     mismatch: int = -1,
                     normalize: bool = False) -> float:
    """Calculate a position-based similarity score.
    
    Compares sequences position by position up to the length of
    the shorter sequence.
    
    Args:
        seq1: First sequence
        seq2: Second sequence
        match: Score for matching positions (default: +1)
        mismatch: Score for mismatching positions (default: -1)
        normalize: If True, divide score by alignment length
    
    Returns:
        Similarity score (raw or normalized)
    """
    # YOUR CODE HERE
    pass

# Test cases:
# similarity_score("ATGC", "ATGC") should return 4
# similarity_score("ATGC", "TTTT") should return -2
# similarity_score("ATGC", "ATGC", normalize=True) should return 1.0

In [ ]:
# Exercise 3: SOLUTION

def similarity_score(seq1: str, seq2: str, 
                     match: int = 1, 
                     mismatch: int = -1,
                     normalize: bool = False) -> float:
    """Calculate a position-based similarity score."""
    seq1, seq2 = seq1.upper(), seq2.upper()
    aligned_len = min(len(seq1), len(seq2))
    
    score = sum(
        match if a == b else mismatch
        for a, b in zip(seq1, seq2)
    )
    
    if normalize and aligned_len > 0:
        return score / aligned_len
    return float(score)

# Verify
assert similarity_score("ATGC", "ATGC") == 4.0
assert similarity_score("ATGC", "TTTT") == -2.0
assert similarity_score("ATGC", "ATGC", normalize=True) == 1.0
print("All tests passed!")

# Compare promoter regions from different species
print(f"\nIdentical seqs:    {similarity_score('ATGCATGC', 'ATGCATGC', normalize=True):.2f}")
print(f"One mutation:      {similarity_score('ATGCATGC', 'ATGAATGC', normalize=True):.2f}")
print(f"Strict scoring:    {similarity_score('ATGCATGC', 'ATGAATGC', match=2, mismatch=-2)}")

### Exercise 4: Codon Frequency Counter

Write a function that counts the frequency of each codon in a DNA sequence. Return a dictionary sorted by frequency (most common first).

In [ ]:
# Exercise 4: Write your function here

def codon_frequency(sequence: str) -> dict:
    """Count frequency of each codon in a DNA sequence.
    
    Reads in frame from position 0.
    
    Args:
        sequence: DNA sequence
    
    Returns:
        dict mapping codons to their counts, sorted by frequency
    """
    # YOUR CODE HERE
    pass

# Test:
# codon_frequency("ATGATGATG") should return {'ATG': 3}
# codon_frequency("ATGAAAGGG") should return {'AAA': 1, 'ATG': 1, 'GGG': 1}

In [ ]:
# Exercise 4: SOLUTION

def codon_frequency(sequence: str) -> dict:
    """Count frequency of each codon in a DNA sequence."""
    sequence = sequence.upper()
    counts = {}
    for i in range(0, len(sequence) - 2, 3):
        codon = sequence[i:i+3]
        if len(codon) == 3:
            counts[codon] = counts.get(codon, 0) + 1
    return dict(sorted(counts.items(), key=lambda x: x[1], reverse=True))

# Test
test_seq = "ATGGCCGATCGATAGCCAATGGATCGATAG"
freq = codon_frequency(test_seq)
print("Codon frequencies:")
for codon, count in freq.items():
    aa = CODON_TABLE.get(codon, '?')
    print(f"  {codon} ({aa}): {count}")

### Exercise 5: ORF Finder with All Six Frames

Extend the ORF finder to search all six reading frames (three on the forward strand, three on the reverse complement).

In [ ]:
# Exercise 5: Write your function here

def find_all_orfs(sequence: str, min_length: int = 30) -> list[dict]:
    """Find ORFs in all six reading frames.
    
    Args:
        sequence: DNA sequence
        min_length: Minimum ORF length in nucleotides
    
    Returns:
        List of dicts with 'start', 'end', 'length', 'strand', 'frame', 'protein' keys
    """
    # YOUR CODE HERE
    # Hint: use the reverse_complement() and find_orfs() functions defined above
    # Search forward strand (frames +1, +2, +3) and reverse strand (frames -1, -2, -3)
    pass

In [ ]:
# Exercise 5: SOLUTION

def find_all_orfs(sequence: str, min_length: int = 30) -> list[dict]:
    """Find ORFs in all six reading frames."""
    all_orfs = []
    
    # Forward strand
    for orf in find_orfs(sequence, min_length):
        orf['strand'] = '+'
        orf['protein'] = translate(orf['sequence'])
        all_orfs.append(orf)
    
    # Reverse strand
    rev_seq = reverse_complement(sequence)
    for orf in find_orfs(rev_seq, min_length):
        orf['strand'] = '-'
        orf['protein'] = translate(orf['sequence'])
        all_orfs.append(orf)
    
    return sorted(all_orfs, key=lambda x: x['length'], reverse=True)

# Test with a sequence that has ORFs on both strands
test_dna = "CCCATGGCCGATCGATAGCCCATGAAAGGGCCCTTTAAATTTCCCATGGCCCCCTGA"
all_orfs = find_all_orfs(test_dna, min_length=9)
print(f"Found {len(all_orfs)} ORFs:")
for orf in all_orfs:
    print(f"  Strand {orf['strand']}, pos {orf['start']}-{orf['end']}: "
          f"{orf['protein']} ({orf['length']} bp)")

---
## Summary

### Key Concepts Covered

| Concept | Syntax | Example |
|---------|--------|---------|
| Function definition | `def name(params):` | `def gc_content(seq):` |
| Return values | `return value` | `return gc_percent` |
| Default arguments | `def f(x=default)` | `def f(table='standard')` |
| Variable positional | `*args` (tuple) | `def f(*sequences)` |
| Variable keyword | `**kwargs` (dict) | `def f(**metadata)` |
| Lambda | `lambda x: expr` | `lambda s: len(s)` |
| Type hints | `def f(x: str) -> int:` | Self-documenting |
| Scope | LEGB rule | Local > Enclosing > Global > Built-in |
| Recursion | Base case + recursive case | Fibonacci, factorial |
| `lru_cache` | `@lru_cache` | Memoize expensive functions |
| `partial` | `partial(f, arg=val)` | Specialize generic functions |

### Best Practices

1. Always write docstrings describing what your function does
2. Avoid mutable default arguments (use `None` instead)
3. Keep functions focused on a single task
4. Use `@lru_cache` for expensive computations with repeated inputs
5. Prefer iterative solutions over recursion for performance-critical code
6. Use type hints to make your code self-documenting

---

**Next module:** File Operations -- reading FASTA, CSV, JSON, and building file-processing pipelines.

---[< Previous: Control Flow for Bioinformatics](../05_Control_Flow/01_control_flow.ipynb) | [Tier 1: Python for Bioinformatics Overview](../README.md) | [Next: File Operations for Bioinformatics >](../07_File_Operations/01_file_operations.ipynb)---